<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Define a baseline

**[Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/)** · Machine Learning: From Problem to Reliable Model · Module 2, lesson 2 of 2 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure three simple baselines on the validation set (the majority class, a constant score and a rule), see why accuracy misleads, and set the improvement target that every model in this course must meet.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | The decision, the capacity of 20% and the two kinds of error, from [Translate a need](https://learning.nextia-ai.com/courses/ml/m02/translate-a-need/). Classification, from [Choose the learning task](https://learning.nextia-ai.com/courses/ml/m01/choose-the-learning-task/). |
| **You do not need** | A local project. The setup below downloads the data. To set up the local project, follow [Set up your project](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/#set-up-your-project) on the lesson page. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M02-L02-define-a-baseline/define-a-baseline-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Baselines on ten tickets

A **baseline** is the simplest prediction that anyone could make without a model, like the forecast "tomorrow will be like today". A model is useful only if it does better than its baselines.

Here are ten made-up tickets. Two of them escalated (T-1 and T-5). Run the cell.

In [ ]:
tiny = pd.DataFrame({
    "ticket": [f"T-{i}" for i in range(1, 11)],
    "priority": [1, 3, 2, 1, 2, 3, 2, 3, 1, 2],
    "team": ["payment", "delivery", "returns", "delivery", "warranty",
             "account", "delivery", "returns", "account", "payment"],
    "escalated": [1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
})
tiny

> **Check.** You should see ten rows. T-1, T-4 and T-9 have priority 1.

Two baselines make a flag for each ticket:

- **Majority class:** always predict the most common class. Here, 8 of 10 tickets did not escalate, so it never flags.
- **Rule:** flag the tickets with priority 1.

The next cell defines `flag_report()`. It compares the flags with the real answers and counts four kinds of result:

| | Escalated | Did not escalate |
|---|---|---|
| **Flagged** | **caught** | **false alarm**: an unnecessary review |
| **Not flagged** | **missed**: a missed escalation | **correctly left** alone |

From the counts, it computes three shares:

- **precision**: of the flagged tickets, the share that really escalated (caught ÷ flagged);
- **recall**: of the escalations, the share that we caught (caught ÷ escalations);
- **accuracy**: of all tickets, the share where the flag was right (caught + correctly left, ÷ all tickets).

[Classification metrics](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/) gives these counts and shares their standard names and explains them fully.

> **Predict.** Which baseline has the higher accuracy on these ten tickets? Work it out by hand, then run the cell.

In [ ]:
def flag_report(y, flags):
    """Compare 0/1 flags with the real 0/1 answers: four counts and four shares."""
    y = np.asarray(y, dtype=bool)
    flags = np.asarray(flags, dtype=bool)
    caught, false_alarms = int((flags & y).sum()), int((flags & ~y).sum())
    missed, correctly_left = int((~flags & y).sum()), int((~flags & ~y).sum())
    return {"flagged": caught + false_alarms, "share flagged": round((caught + false_alarms) / len(y), 3),
            "caught": caught, "false alarms": false_alarms, "missed": missed, "correctly left": correctly_left,
            "precision": round(caught / (caught + false_alarms), 3) if caught + false_alarms else None,
            "recall": round(caught / (caught + missed), 3), "accuracy": round((caught + correctly_left) / len(y), 3)}


pd.set_option("display.width", 140)  # show wide tables on one line
pd.DataFrame([flag_report(tiny["escalated"], [0] * 10),
              flag_report(tiny["escalated"], tiny["priority"] == 1)],
             index=["majority", "rule: priority 1"])

> **Check.** You should see, for the majority: 0 flagged, precision `NaN` (it flags nothing, so there is nothing to divide), recall `0.0`, accuracy `0.8`. For the rule: 3 flagged, 1 caught, 2 false alarms, 1 missed, precision `0.333`, recall `0.5`, accuracy `0.7`.

Check it by hand. The majority is right on the 8 tickets that did not escalate: 8 ÷ 10 = 0.8. It catches neither escalation. The rule flags T-1, T-4 and T-9. Only T-1 escalated: precision 1 ÷ 3 = 0.333. It catches T-1 but not T-5: recall 1 ÷ 2 = 0.5. It is right on 7 tickets: T-1 and the 6 tickets that it correctly does not flag.

The useless baseline has the **higher** accuracy. Remember this.

## 2. A constant score

A model in this course gives each ticket a **score** from 0 to 1, and the senior team reviews the tickets with the highest scores. The score baseline gives every ticket the same score: the escalation rate. Two measures judge a score without a threshold ([The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/) explains them):

- **ROC AUC:** the chance that a random escalated ticket gets a higher score than a random other ticket. 0.5 is guessing; 1.0 is a perfect ranking.
- **Average precision (AP):** the mean precision as you go down the ranked list. For a score that cannot rank, it equals the escalation rate.

Run the cell. It scores the ten tickets with 0.2, their escalation rate.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

tiny_scores = [0.2] * 10
print("ROC AUC:", roc_auc_score(tiny["escalated"], tiny_scores))
print("Average precision:", average_precision_score(tiny["escalated"], tiny_scores))

> **Check.** You should see `ROC AUC: 0.5` and `Average precision: 0.2`.

Every ticket has the same score, so the score cannot put an escalated ticket above another one. That is the floor: a model's ROC AUC must be above 0.5, and its AP above the escalation rate.

## 3. The baselines on the real data

Now the real validation set: the 1,179 tickets of May. Every baseline learns only from train. The majority class and the constant score come from train; the rule needs no data.

Run the next cell. It loads train and valid and finds the most common class and the escalation rate of train.

In [ ]:
def read(path):
    """Read a ticket file. Only an empty cell is missing: "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


TARGET = "escalated_72h"
train = read("data/train.csv")
valid = read("data/valid.csv")
majority_class = train[TARGET].mode()[0]
train_rate = round(train[TARGET].mean(), 3)
print("Most common class in train:", majority_class, "  train escalation rate:", train_rate)

> **Check.** You should see `Most common class in train: 0` and `train escalation rate: 0.117`.

> **Predict.** The rule flagged 3 of 10 tiny tickets. On the 1,179 May tickets, will it flag more or less than 20%? Which baseline will have the higher accuracy? Then run the cell.

In [ ]:
y = valid[TARGET]
baselines = pd.DataFrame([flag_report(y, [majority_class] * len(valid)),
                          flag_report(y, valid["priority"] == 1)],
                         index=["majority: never escalate", "rule: flag priority 1"])
baselines

> **Check.** You should see, for the majority: 0 flagged, precision `NaN`, recall `0.000`, accuracy `0.845`. For the rule: `163` flagged (`0.138`), 61 caught, 102 false alarms, 122 missed, precision `0.374`, recall `0.333`, accuracy `0.810`.

The rule flags 13.8% of the tickets, inside the capacity of 20%. Of its flags, 37.4% escalated, and it catches one escalation in three. The majority baseline is right on 84.5% of the tickets and catches nothing.

Run the next cell for the constant score on valid.

In [ ]:
constant_scores = [train_rate] * len(valid)
print("ROC AUC:", roc_auc_score(y, constant_scores))
print("Average precision:", round(average_precision_score(y, constant_scores), 3))
print("Escalation rate in valid:", round(y.mean(), 3))

> **Check.** You should see `ROC AUC: 0.5`, `Average precision: 0.155` and `Escalation rate in valid: 0.155`.

## 4. The improvement target

Grace's decision gives two conditions. The senior team can review at most 20% of new tickets, and the rule already catches 33%. So Priya writes the **improvement target**:

> Flag at most 20% of new tickets, and catch at least 40% of escalations.

Run the cell. It defines `meets_target()` and tests the rule.

In [ ]:
def meets_target(report):
    """True if the flags fit the capacity (at most 20%) and catch at least 40%."""
    return report["share flagged"] <= 0.20 and report["recall"] >= 0.40


rule = flag_report(y, valid["priority"] == 1)
print("Rule meets the target:", meets_target(rule))

> **Check.** You should see `Rule meets the target: False`. The rule fits the capacity, but its recall is 0.333.

## 5. Your turn: Grace's rule

Change one thing: the rule. Grace says: "Since the new warranty policy, warranty tickets escalate more. Flag priority 1 **or** warranty tickets."

> **Predict.** Will Grace's rule catch more escalations than the priority-1 rule? Will it still fit the capacity?

> **Your turn.** Make `grace_flags`: True for each May ticket with priority 1 or `team` equal to `"warranty"` (use `|` for "or"). Then put `flag_report(y, grace_flags)` in `grace`. Run the cell, then the check cell.

In [ ]:
grace_flags = ...  # priority 1 or team "warranty"
grace = ...  # the flag report of grace_flags
print(grace)
print("Meets the target:", meets_target(grace) if isinstance(grace, dict) else "?")

In [ ]:
expect_hash("your report of Grace's rule", [grace["flagged"], grace["precision"], grace["recall"]] if isinstance(grace, dict) else None, 'a8b2d6990bd55742')

Grace's rule catches more than half of the escalations. It does not meet the target, because it flags 26.5% of the tickets, more than the senior team can review. A rule has no knob to move the share: it flags what it flags. A score can be cut at any level. [The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/) shows how, and [Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/) chooses the cut that fits the 20%.

**Optional:** write your own rule in the next cell and run it. You can combine conditions with `&` (and) and `|` (or). Note your rule and its numbers for the module practice. Do not try many rules here: each look at the May tickets makes the May numbers less honest ([Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/) explains why).

In [ ]:
my_flags = (valid["priority"] == 1)  # change this rule
mine = flag_report(y, my_flags)
print(mine)
print("Meets the target:", meets_target(mine))

## 6. Failure case: accuracy picks the useless baseline

Tomás compares the three rules by accuracy, because "accuracy is the standard measure". Run the cell.

> **Predict.** Which rule has the highest accuracy? Which one catches the most escalations?

In [ ]:
compare = pd.DataFrame([flag_report(y, [0] * len(valid)),
                        flag_report(y, valid["priority"] == 1),
                        flag_report(y, (valid["priority"] == 1) | (valid["team"] == "warranty"))],
                       index=["never escalate", "priority 1", "priority 1 or warranty"])
compare[["flagged", "recall", "accuracy"]].sort_values("accuracy", ascending=False)

> **Check.** You should see `never escalate` first, with accuracy `0.845` and recall `0.0`. `priority 1 or warranty` is last, with accuracy `0.746` and the highest recall, `0.536`.

The order by accuracy is the opposite of the order by escalations caught. With 15.5% escalations, a rule that says "no" every time is right 84.5% of the time. Accuracy counts every correct "no", and the decision is about the "yes". So the measures for this course are the share flagged (against the capacity) and the recall and precision of the flags; for scores, AP and ROC AUC. [Classification metrics](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/) explains this trap in full.

## 7. Module practice: the modelling brief

Write a one-page modelling brief for ticket escalation, before any model exists. The [lesson page](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/#module-practice) has the template and Priya's example. Double-click the next cell and fill it in. Use the numbers from this module.

**Modelling brief: ticket escalation**

| Part | Your answer |
|---|---|
| Decision and owner | |
| Prediction unit | |
| Prediction time | |
| Target | |
| Features | |
| Never features | |
| Data and split | |
| Cost of a missed escalation | |
| Cost of an unnecessary review | |
| Capacity | |
| Baselines (on valid) | |
| Measures | |
| Improvement target | |
| Why not only a rule | |

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/#module-check). They count toward your certificate when you are signed in and enrolled.

In this lesson, you measured three baselines on May. Never escalating is right on 84.5% of the tickets and catches nothing. A constant score has ROC AUC 0.5 and AP 0.155. The rule "flag priority 1" flags 13.8%, with precision 0.374 and recall 0.333. Grace's rule catches 53.6% but flags 26.5%, above the capacity. The improvement target for every model in this course: flag at most 20% and catch at least 40%.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.